In [1]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ──────────────────────────────────────────────────────────────
# Đổi DATA_DIR thành folder chứa CSV của bạn
DATA_DIR = "/kaggle/input/competitions/datathon-2026-round-1"
NATURAL_NULL_VALUES = {
    'nan', 'NaN', 'null', 'NULL', 'none', 'None',
    '.', '-', '--', 'N/A', 'n/a', 'NA', '',  ' '
}
HIGH_CARD_THRESHOLD = 0.95          # Tỉ lệ unique/total > 95% → likely PK/FK
LOW_CARD_THRESHOLD  = 0.05          # Tỉ lệ unique/total < 5%  → likely category
NULL_WARN_THRESHOLD = 0.20          # Null > 20% → cảnh báo
# ────────────────────────────────────────────────────────────────────────

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "**/*.csv"), recursive=True) +
                   glob.glob(os.path.join(DATA_DIR, "*.csv")))
csv_files = list(dict.fromkeys(csv_files))  # deduplicate

print(f"📂 DATA_DIR : {os.path.abspath(DATA_DIR)}")
print(f"📄 CSV files: {len(csv_files)} found")
for f in csv_files:
    size_kb = os.path.getsize(f) / 1024
    print(f"   • {Path(f).name:40s}  {size_kb:8.1f} KB")

📂 DATA_DIR : /kaggle/input/competitions/datathon-2026-round-1
📄 CSV files: 14 found
   • customers.csv                               6913.7 KB
   • geography.csv                               1369.4 KB
   • inventory.csv                               5535.8 KB
   • order_items.csv                            23381.6 KB
   • orders.csv                                 44886.0 KB
   • payments.csv                               17952.4 KB
   • products.csv                                 190.6 KB
   • promotions.csv                                 4.3 KB
   • returns.csv                                 2228.0 KB
   • reviews.csv                                 6632.2 KB
   • sales.csv                                    126.6 KB
   • sample_submission.csv                         18.1 KB
   • shipments.csv                              19293.0 KB
   • web_traffic.csv                              203.8 KB


In [2]:
# ── LOAD ALL CSVs ────────────────────────────────────────────────────────
dfs = {}
load_errors = {}

for path in csv_files:
    name = Path(path).stem
    try:
        # Thử detect encoding + đọc sample
        df = pd.read_csv(path)
        dfs[name] = df
        total_rows = sum(1 for _ in open(path, encoding='utf-8', errors='replace')) - 1
        sampled = len(df) < total_rows
        flag = f" (sampled {len(df):,}/{total_rows:,})" if sampled else ""
        print(f"✅ {name:30s}  {len(df):>7,} rows × {len(df.columns):>3} cols{flag}")
    except Exception as e:
        load_errors[name] = str(e)
        print(f"❌ {name}: {e}")

print(f"\nLoaded {len(dfs)}/{len(csv_files)} tables successfully.")

✅ customers                       121,930 rows ×   7 cols
✅ geography                        39,948 rows ×   4 cols
✅ inventory                        60,247 rows ×  17 cols
✅ order_items                     714,669 rows ×   7 cols
✅ orders                          646,945 rows ×   8 cols
✅ payments                        646,945 rows ×   4 cols
✅ products                          2,412 rows ×   8 cols
✅ promotions                           50 rows ×  10 cols
✅ returns                          39,939 rows ×   7 cols
✅ reviews                         113,551 rows ×   7 cols
✅ sales                             3,833 rows ×   3 cols
✅ sample_submission                   548 rows ×   3 cols
✅ shipments                       566,067 rows ×   4 cols
✅ web_traffic                       3,652 rows ×   7 cols

Loaded 14/14 tables successfully.


# 1. Schema & dtypes

In [3]:
def dtype_category(dtype):
    """Map pandas dtype → readable category."""
    s = str(dtype)
    if 'int' in s:   return 'integer'
    if 'float' in s: return 'float'
    if 'bool' in s:  return 'boolean'
    if 'datetime' in s: return 'datetime'
    return 'string/object'

all_schemas = []

for table, df in dfs.items():
    for col in df.columns:
        all_schemas.append({
            'table':   table,
            'column':  col,
            'dtype':   str(df[col].dtype),
            'type':    dtype_category(df[col].dtype),
            'n_rows':  len(df),
        })

schema_df = pd.DataFrame(all_schemas)

# Display per table
for table, grp in schema_df.groupby('table'):
    print(f"\n{'═'*60}")
    print(f"  TABLE: {table.upper()}   ({grp['n_rows'].iloc[0]:,} rows × {len(grp)} cols)")
    print(f"{'═'*60}")
    display(grp[['column','dtype','type']].reset_index(drop=True).style
            .set_table_styles([{'selector':'th','props':[('background','#2d2d2d'),('color','#fff'),('padding','6px 12px')]}])
            .apply(lambda col: ['background: #fff9db' if v == 'string/object' else '' for v in col], subset=['type']))


════════════════════════════════════════════════════════════
  TABLE: CUSTOMERS   (121,930 rows × 7 cols)
════════════════════════════════════════════════════════════


,column,dtype,type
0,customer_id,int64,integer
1,zip,int64,integer
2,city,object,string/object
3,signup_date,object,string/object
4,gender,object,string/object
5,age_group,object,string/object
6,acquisition_channel,object,string/object



════════════════════════════════════════════════════════════
  TABLE: GEOGRAPHY   (39,948 rows × 4 cols)
════════════════════════════════════════════════════════════


,column,dtype,type
0,zip,int64,integer
1,city,object,string/object
2,region,object,string/object
3,district,object,string/object



════════════════════════════════════════════════════════════
  TABLE: INVENTORY   (60,247 rows × 17 cols)
════════════════════════════════════════════════════════════


...

,column,dtype,type
0,Date,object,string/object
1,Revenue,float64,float
2,COGS,float64,float



════════════════════════════════════════════════════════════
  TABLE: SHIPMENTS   (566,067 rows × 4 cols)
════════════════════════════════════════════════════════════


,column,dtype,type
0,order_id,int64,integer
1,ship_date,object,string/object
2,delivery_date,object,string/object
3,shipping_fee,float64,float



════════════════════════════════════════════════════════════
  TABLE: WEB_TRAFFIC   (3,652 rows × 7 cols)
════════════════════════════════════════════════════════════


,column,dtype,type
0,date,object,string/object
1,sessions,int64,integer
2,unique_visitors,int64,integer
3,page_views,int64,integer
4,bounce_rate,float64,float
5,avg_session_duration_sec,float64,float
6,traffic_source,object,string/object


# 2. Null % per Column

In [4]:
# ── NORMALIZE NATURAL NULLS ──────────────────────────────────────────────
import re

def normalize_nulls(df: pd.DataFrame, null_values: set) -> tuple[pd.DataFrame, dict]:
    """
    Với mỗi column kiểu object/string:
      1. Strip whitespace đầu/cuối
      2. So sánh (case-sensitive) với null_values
      3. Thay thế bằng np.nan
    Trả về (df_normalized, report) trong đó report ghi lại
    số ô được chuyển đổi per column.
    """
    df = df.copy()
    report = {}  # col → n_converted

    for col in df.select_dtypes(include='object').columns:
        # Strip whitespace (tạo series tạm, không mutate ngay)
        stripped = df[col].str.strip()

        # Mask: stripped value nằm trong null_values (và chưa phải NaN)
        mask = stripped.isin(null_values) & df[col].notna()
        n_converted = int(mask.sum())

        if n_converted > 0:
            df.loc[mask, col] = np.nan
            report[col] = n_converted

    return df, report


# ── Áp dụng lên toàn bộ dfs (in-place update) ───────────────────────────
normalize_summary = []   # dùng cho report
total_converted   = 0

for table in list(dfs.keys()):
    before_nulls = dfs[table].isnull().sum().sum()
    dfs[table], col_report = normalize_nulls(dfs[table], NATURAL_NULL_VALUES)
    after_nulls  = dfs[table].isnull().sum().sum()
    delta        = int(after_nulls - before_nulls)
    total_converted += delta

    normalize_summary.append({
        'table':           table,
        'null_before':     int(before_nulls),
        'null_after':      int(after_nulls),
        'cells_converted': delta,
        'cols_affected':   list(col_report.keys()),
    })

# ── Print report ─────────────────────────────────────────────────────────
print(f"Null normalization — pattern list ({len(NATURAL_NULL_VALUES)} patterns):")
print(' | '.join(sorted(repr(v) for v in NATURAL_NULL_VALUES)))
print()

any_change = False
for row in normalize_summary:
    changed = row['cells_converted'] > 0
    icon    = '🔄' if changed else '✅'
    print(f"{icon} {row['table']:30s}  "
          f"null: {row['null_before']:>6,} → {row['null_after']:>6,}  "
          f"(+{row['cells_converted']:,} cells converted)")
    if changed:
        any_change = True
        for col in row['cols_affected']:
            n = dfs[row['table']][col].isnull().sum()
            print(f"   └─ {col:28s}  null sau normalize: {n:,}")

print()
if total_converted == 0:
    print("✅ Không tìm thấy giá trị thiếu tự nhiên nào. dfs giữ nguyên.")
else:
    print(f"⚠️  Tổng {total_converted:,} ô đã được chuẩn hóa → np.nan.")
    print("   Các bước 2 (Null %), 3 (Cardinality), 4 (FK) sẽ dùng dfs đã normalize.")

Null normalization — pattern list (14 patterns):
' ' | '' | '-' | '--' | '.' | 'N/A' | 'NA' | 'NULL' | 'NaN' | 'None' | 'n/a' | 'nan' | 'none' | 'null'

✅ customers                       null:      0 →      0  (+0 cells converted)
✅ geography                       null:      0 →      0  (+0 cells converted)
✅ inventory                       null:      0 →      0  (+0 cells converted)
✅ order_items                     null: 1,152,816 → 1,152,816  (+0 cells converted)
✅ orders                          null:      0 →      0  (+0 cells converted)
✅ payments                        null:      0 →      0  (+0 cells converted)
✅ products                        null:      0 →      0  (+0 cells converted)
✅ promotions                      null:     40 →     40  (+0 cells converted)
✅ returns                         null:      0 →      0  (+0 cells converted)
✅ reviews                         null:      0 →      0  (+0 cells converted)
✅ sales                           null:      0 →      0  (+0 

In [5]:
null_records = []

for table, df in dfs.items():
    null_counts = df.isnull().sum()
    null_pct    = df.isnull().mean() * 100
    for col in df.columns:
        null_records.append({
            'table':      table,
            'column':     col,
            'null_count': int(null_counts[col]),
            'null_%':     round(float(null_pct[col]), 2),
            'warning':    '⚠️ HIGH NULL' if null_pct[col] > NULL_WARN_THRESHOLD * 100 else ''
        })

null_df = pd.DataFrame(null_records)

# Summary: columns with any nulls
has_nulls = null_df[null_df['null_count'] > 0].copy()

if has_nulls.empty:
    print("✅ No null values detected in any table!")
else:
    print(f"⚠️  {len(has_nulls)} columns have null values:\n")
    
    def null_bar(pct, width=20):
        filled = int(round(pct / 100 * width))
        return '█' * filled + '░' * (width - filled) + f'  {pct:.1f}%'
    
    has_nulls['bar'] = has_nulls['null_%'].apply(null_bar)
    
    for table, grp in has_nulls.groupby('table'):
        print(f"\n── {table} ──")
        for _, row in grp.sort_values('null_%', ascending=False).iterrows():
            warn = row['warning']
            print(f"  {row['column']:30s}  {row['bar']}  ({row['null_count']:,} rows) {warn}")

print(f"\n{'─'*60}")
print("Full null summary:")
display(null_df.sort_values(['table','null_%'], ascending=[True, False])
              .reset_index(drop=True))

⚠️  3 columns have null values:


── order_items ──
  promo_id_2                      ████████████████████  100.0%  (714,463 rows) ⚠️ HIGH NULL
  promo_id                        ████████████░░░░░░░░  61.3%  (438,353 rows) ⚠️ HIGH NULL

── promotions ──
  applicable_category             ████████████████░░░░  80.0%  (40 rows) ⚠️ HIGH NULL

────────────────────────────────────────────────────────────
Full null summary:


,table,column,null_count,null_%,warning
0,customers,customer_id,0,0.0,
1,customers,zip,0,0.0,
2,customers,city,0,0.0,
3,customers,signup_date,0,0.0,
4,customers,gender,0,0.0,
...,...,...,...,...,...
91,web_traffic,unique_visitors,0,0.0,
92,web_traffic,page_views,0,0.0,
93,web_traffic,bounce_rate,0,0.0,
94,web_traffic,avg_session_duration_sec,0,0.0,


# 3. Cardinality

In [6]:
card_records = []

for table, df in dfs.items():
    n = len(df)
    for col in df.columns:
        n_unique = df[col].nunique(dropna=True)
        ratio    = n_unique / n if n > 0 else 0
        
        if ratio > HIGH_CARD_THRESHOLD:
            card_type = '🔑 high (PK/FK candidate)'
        elif ratio < LOW_CARD_THRESHOLD:
            card_type = '🏷️  low (category/enum)'
        else:
            card_type = '📊 medium'
        
        # Sample top values for low-cardinality cols
        if ratio < LOW_CARD_THRESHOLD and n_unique <= 20:
            top_vals = df[col].value_counts().head(5)
            sample = ', '.join([f"{v}({c})" for v,c in top_vals.items()])
        else:
            sample = '—'
        
        card_records.append({
            'table':       table,
            'column':      col,
            'n_unique':    n_unique,
            'unique_ratio': round(ratio, 4),
            'cardinality': card_type,
            'top_values':  sample,
        })

card_df = pd.DataFrame(card_records)

print("Cardinality Summary\n")

for table, grp in card_df.groupby('table'):
    print(f"\n── {table} ({dfs[table].shape[0]:,} rows) ──")
    display(grp[['column','n_unique','unique_ratio','cardinality','top_values']]
            .reset_index(drop=True))

Cardinality Summary


── customers (121,930 rows) ──


,column,n_unique,unique_ratio,cardinality,top_values
0,customer_id,121930,1.0000,🔑 high (PK/FK candidate),—
1,zip,31491,0.2583,📊 medium,—
2,city,42,0.0003,🏷️ low (category/enum),—
3,signup_date,3941,0.0323,🏷️ low (category/enum),—
4,gender,3,0.0000,🏷️ low (category/enum),"Female(59640), Male(57457), Non-binary(4833)"
5,age_group,5,0.0000,🏷️ low (category/enum),"25-34(36342), 35-44(31920), 45-54(23172), 18-2..."
6,acquisition_channel,6,0.0000,🏷️ low (category/enum),"organic_search(36450), social_media(24448), pa..."



── geography (39,948 rows) ──


,column,n_unique,unique_ratio,cardinality,top_values
0,zip,39948,1.0000,🔑 high (PK/FK candidate),—
1,city,42,0.0011,🏷️ low (category/enum),—
2,region,3,0.0001,🏷️ low (category/enum),"East(18929), Central(14512), West(6507)"
3,district,39,0.0010,🏷️ low (category/enum),—



── inventory (60,247 rows) ──


...

,column,n_unique,unique_ratio,cardinality,top_values
0,Date,548,1.0,🔑 high (PK/FK candidate),—
1,Revenue,548,1.0,🔑 high (PK/FK candidate),—
2,COGS,548,1.0,🔑 high (PK/FK candidate),—



── shipments (566,067 rows) ──


,column,n_unique,unique_ratio,cardinality,top_values
0,order_id,566067,1.0000,🔑 high (PK/FK candidate),—
1,ship_date,3831,0.0068,🏷️ low (category/enum),—
2,delivery_date,3831,0.0068,🏷️ low (category/enum),—
3,shipping_fee,1856,0.0033,🏷️ low (category/enum),—



── web_traffic (3,652 rows) ──


,column,n_unique,unique_ratio,cardinality,top_values
0,date,3652,1.0000,🔑 high (PK/FK candidate),—
1,sessions,3447,0.9439,📊 medium,—
2,unique_visitors,3382,0.9261,📊 medium,—
3,page_views,3620,0.9912,🔑 high (PK/FK candidate),—
4,bounce_rate,261,0.0715,📊 medium,—
5,avg_session_duration_sec,1771,0.4849,📊 medium,—
6,traffic_source,6,0.0016,🏷️ low (category/enum),"organic_search(1090), paid_search(784), social..."


# 4. Foreign Key Detection

In [7]:
# ── STEP 4a: Detect Primary Key candidates ───────────────────────────────
pk_candidates = {}   # table → [col, ...]

for table, df in dfs.items():
    pks = []
    for col in df.columns:
        n_unique = df[col].nunique(dropna=True)
        n_null   = df[col].isnull().sum()
        ratio    = n_unique / len(df) if len(df) > 0 else 0
        # PK: fully unique, no nulls
        if n_unique == len(df) and n_null == 0:
            pks.append(col)
    pk_candidates[table] = pks

print("🔑 Primary Key Candidates:")
for table, pks in pk_candidates.items():
    pk_str = ', '.join(pks) if pks else '(none found)'
    print(f"   {table:30s} → {pk_str}")

# ── STEP 4b: Detect FK candidates ───────────────────────────────────────
print("\n" + "─"*60)
print("🔗 Foreign Key Candidates (col values ⊆ PK values in another table):")
print("─"*60)

fk_links = []

for src_table, src_df in dfs.items():
    for src_col in src_df.columns:
        src_vals = set(src_df[src_col].dropna().unique())
        if len(src_vals) == 0:
            continue
        
        for ref_table, ref_pks in pk_candidates.items():
            if ref_table == src_table:
                continue
            for ref_pk in ref_pks:
                ref_vals = set(dfs[ref_table][ref_pk].dropna().unique())
                if len(ref_vals) == 0:
                    continue
                
                # Check: src values are subset of ref PK values
                coverage = len(src_vals & ref_vals) / len(src_vals)
                
                if coverage >= 0.80:  # 80%+ match → likely FK
                    fk_links.append({
                        'from_table':  src_table,
                        'from_col':    src_col,
                        'to_table':    ref_table,
                        'to_col':      ref_pk,
                        'coverage_%':  round(coverage * 100, 1),
                        'src_unique':  len(src_vals),
                        'ref_unique':  len(ref_vals),
                    })

if fk_links:
    fk_df = pd.DataFrame(fk_links).sort_values('coverage_%', ascending=False)
    
    for _, row in fk_df.iterrows():
        bar_len = int(row['coverage_%'] / 5)
        bar = '█' * bar_len + '░' * (20 - bar_len)
        print(f"  {row['from_table']}.{row['from_col']:20s}  →  "
              f"{row['to_table']}.{row['to_col']:15s}  "
              f"{bar} {row['coverage_%']}%")
    
    print("\nFull FK table:")
    display(fk_df.reset_index(drop=True))
else:
    print("  No FK candidates found (try lowering the coverage threshold).")

🔑 Primary Key Candidates:
   customers                      → customer_id
   geography                      → zip
   inventory                      → (none found)
   order_items                    → (none found)
   orders                         → order_id
   payments                       → order_id
   products                       → product_id
   promotions                     → promo_id, promo_name, start_date, end_date
   returns                        → return_id
   reviews                        → review_id
   sales                          → Date, Revenue, COGS
   sample_submission              → Date, Revenue, COGS
   shipments                      → order_id
   web_traffic                    → date

────────────────────────────────────────────────────────────
🔗 Foreign Key Candidates (col values ⊆ PK values in another table):
────────────────────────────────────────────────────────────
  customers.zip                   →  geography.zip              ████████████████████ 100.0%

,from_table,from_col,to_table,to_col,coverage_%,src_unique,ref_unique
0,customers,zip,geography,zip,100.0,31491,39948
1,inventory,snapshot_date,sales,Date,100.0,126,3833
2,inventory,units_sold,products,product_id,100.0,303,2412
3,inventory,units_received,products,product_id,100.0,360,2412
4,inventory,product_id,products,product_id,100.0,1624,2412
...,...,...,...,...,...,...,...
80,payments,installments,shipments,order_id,80.0,5,566067
81,payments,installments,orders,order_id,80.0,5,646945
82,reviews,rating,shipments,order_id,80.0,5,566067
83,reviews,rating,orders,order_id,80.0,5,646945


# 5. Entity Relationship Diagram

In [8]:
# Print Mermaid ERD bạn có thể paste vào https://mermaid.live
lines = ["```mermaid", "erDiagram"]

# Tables with columns
for table, df in dfs.items():
    lines.append(f"  {table.upper()} {{")
    for col in df.columns:
        dtype = dtype_category(str(df[col].dtype))
        # Mark PKs
        pk_mark = " PK" if col in pk_candidates.get(table, []) else ""
        lines.append(f"    {dtype} {col}{pk_mark}")
    lines.append("  }")

# Relationships from FK detection
if fk_links:
    added = set()
    for link in fk_links:
        key = (link['from_table'], link['to_table'])
        if key not in added:
            lines.append(f"  {link['to_table'].upper()} ||--o{{ {link['from_table'].upper()} : \"{link['to_col']} → {link['from_col']}\"")
            added.add(key)

lines.append("```")
mermaid_src = "\n".join(lines)

print("Paste vào https://mermaid.live để xem ERD diagram:\n")
print(mermaid_src)

Paste vào https://mermaid.live để xem ERD diagram:

```mermaid
erDiagram
  CUSTOMERS {
    integer customer_id PK
    integer zip
    string/object city
    string/object signup_date
    string/object gender
    string/object age_group
    string/object acquisition_channel
  }
  GEOGRAPHY {
    integer zip PK
    string/object city
    string/object region
    string/object district
  }
  INVENTORY {
    string/object snapshot_date
    integer product_id
    integer stock_on_hand
    integer units_received
    integer units_sold
    integer stockout_days
    float days_of_supply
    float fill_rate
    integer stockout_flag
    integer overstock_flag
    integer reorder_flag
    float sell_through_rate
    string/object product_name
    string/object category
    string/object segment
    integer year
    integer month
  }
  ORDER_ITEMS {
    integer order_id
    integer product_id
    integer quantity
    float unit_price
    float discount_amount
    string/object promo_id
    string

# 6. Master Summary

In [9]:
summary_rows = []

for table, df in dfs.items():
    total_nulls = df.isnull().sum().sum()
    total_cells = df.shape[0] * df.shape[1]
    null_pct_overall = total_nulls / total_cells * 100
    
    dtype_counts = pd.Series([dtype_category(str(df[c].dtype)) for c in df.columns]).value_counts().to_dict()
    
    pk_cols = pk_candidates.get(table, [])
    fk_cols = [l['from_col'] for l in fk_links if l['from_table'] == table]
    
    summary_rows.append({
        'table':         table,
        'rows':          f"{len(df):,}",
        'cols':          len(df.columns),
        'null_%':        f"{null_pct_overall:.1f}%",
        'pk_cols':       ', '.join(pk_cols) or '—',
        'fk_cols':       ', '.join(set(fk_cols)) or '—',
        'dtypes':        ' | '.join([f"{k}×{v}" for k,v in dtype_counts.items()]),
    })

summary = pd.DataFrame(summary_rows)

print("="*80)
print("MASTER SCHEMA SUMMARY")
print("="*80)
display(summary.style
        .set_table_styles([{
            'selector': 'th',
            'props': [('background-color','#1a1a2e'),('color','white'),('padding','8px 14px'),('font-size','13px')]
        },{
            'selector': 'td',
            'props': [('padding','6px 12px'),('font-size','13px')]
        }])
        .set_caption("Auto-generated by 00_schema_overview.ipynb")
)

MASTER SCHEMA SUMMARY


,table,rows,cols,null_%,pk_cols,fk_cols,dtypes
0,customers,"121,930",7,0.0%,customer_id,"zip, signup_date",string/object×5 | integer×2
1,geography,"39,948",4,0.0%,zip,—,string/object×3 | integer×1
2,inventory,"60,247",17,0.0%,—,"year, product_id, snapshot_date, units_received, month, stockout_days, units_sold, stock_on_hand",integer×10 | string/object×4 | float×3
3,order_items,"714,669",7,23.0%,—,"product_id, promo_id_2, quantity, promo_id, order_id",integer×3 | float×2 | string/object×2
4,orders,"646,945",8,0.0%,order_id,"zip, order_date, order_id, customer_id",string/object×5 | integer×3
5,payments,"646,945",4,0.0%,order_id,"installments, order_id",integer×2 | string/object×1 | float×1
6,products,"2,412",8,0.0%,product_id,—,string/object×5 | float×2 | integer×1
7,promotions,50,10,8.0%,"promo_id, promo_name, start_date, end_date","start_date, end_date, discount_value",string/object×7 | integer×2 | float×1
8,returns,"39,939",7,0.0%,return_id,"return_quantity, product_id, order_id, return_date",string/object×3 | integer×3 | float×1
9,reviews,"113,551",7,0.0%,review_id,"product_id, rating, review_date, order_id, customer_id",integer×4 | string/object×3
